In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional,SimpleRNN
from tensorflow.keras.models import Sequential


In [ ]:
data=pd.read_csv('Sentiment Analysis Dataset.csv',encoding='latin1')

In [ ]:
data

,id,sentiment,date,text,Unnamed: 4,Unnamed: 5
0,623495523,1,Mon Dec 01 20:46:01 +0000 2014,WTF MY BATTERY WAS 31% ONE SECOND AGO AND NOW ...,NaN,NaN
1,623495527,1,Mon Dec 01 21:09:50 +0000 2014,@apple Contact sync between Yosemite and iOS8 ...,NaN,NaN
2,623495529,1,Mon Dec 01 21:35:14 +0000 2014,WARNING IF YOU BUY AN IPHONE 5S UNLOCKED FROM ...,NaN,NaN
3,623495536,1,Mon Dec 01 23:55:55 +0000 2014,"@Apple, For the love of GAWD, CENTER the '1'on...",NaN,NaN
4,623495537,1,Tue Dec 02 00:06:05 +0000 2014,i get the storage almost full notification lit...,NaN,NaN
...,...,...,...,...,...,...
3881,623499108,not_relevant,Tue Dec 09 08:30:04 +0000 2014,#AAPL:Accel Partners Leads $50M Series C Fundi...,NaN,NaN
3882,623499141,not_relevant,Tue Dec 09 11:52:37 +0000 2014,Counting down the minutes! Interest in full ti...,NaN,NaN
3883,623499180,not_relevant,Tue Dec 09 14:13:17 +0000 2014,@JustinPulitzer Any comment on #aapl today?,NaN,NaN
3884,623499194,not_relevant,Tue Dec 09 14:39:59 +0000 2014,Have been brave and taken out an #AAPL CFD as ...,NaN,NaN


In [ ]:
data['text']

0       WTF MY BATTERY WAS 31% ONE SECOND AGO AND NOW ...
1       @apple Contact sync between Yosemite and iOS8 ...
2       WARNING IF YOU BUY AN IPHONE 5S UNLOCKED FROM ...
3       @Apple, For the love of GAWD, CENTER the '1'on...
4       i get the storage almost full notification lit...
                              ...                        
3881    #AAPL:Accel Partners Leads $50M Series C Fundi...
3882    Counting down the minutes! Interest in full ti...
3883          @JustinPulitzer Any comment on #aapl today?
3884    Have been brave and taken out an #AAPL CFD as ...
3885    Tim Cook Met With Jesse Jackson for 'Positive ...
Name: text, Length: 3886, dtype: object

In [ ]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3886 entries, 0 to 3885
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          3886 non-null   int64  
 1   sentiment   3886 non-null   object 
 2   date        3886 non-null   object 
 3   text        3886 non-null   object 
 4   Unnamed: 4  0 non-null      float64
 5   Unnamed: 5  0 non-null      float64
dtypes: float64(2), int64(1), object(3)
memory usage: 182.3+ KB
None


In [ ]:
data['Unnamed: 4'].head(5)

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: Unnamed: 4, dtype: float64

In [ ]:
print(data['Unnamed: 4'].value_counts())

Series([], Name: Unnamed: 4, dtype: int64)


In [ ]:
#DATA CLEANING


In [ ]:
data=data[['sentiment','text']]
data

,sentiment,text
0,1,WTF MY BATTERY WAS 31% ONE SECOND AGO AND NOW ...
1,1,@apple Contact sync between Yosemite and iOS8 ...
2,1,WARNING IF YOU BUY AN IPHONE 5S UNLOCKED FROM ...
3,1,"@Apple, For the love of GAWD, CENTER the '1'on..."
4,1,i get the storage almost full notification lit...
...,...,...
3881,not_relevant,#AAPL:Accel Partners Leads $50M Series C Fundi...
3882,not_relevant,Counting down the minutes! Interest in full ti...
3883,not_relevant,@JustinPulitzer Any comment on #aapl today?
3884,not_relevant,Have been brave and taken out an #AAPL CFD as ...


In [ ]:
data=data[data['sentiment']!='not_relevant']
data=data[data['sentiment']!='3']
print(data)


     sentiment                                               text
0            1  WTF MY BATTERY WAS 31% ONE SECOND AGO AND NOW ...
1            1  @apple Contact sync between Yosemite and iOS8 ...
2            1  WARNING IF YOU BUY AN IPHONE 5S UNLOCKED FROM ...
3            1  @Apple, For the love of GAWD, CENTER the '1'on...
4            1  i get the storage almost full notification lit...
...        ...                                                ...
3799         5  Lets go shopping!!! @apple http://t.co/uTFUHuoJIi
3800         5  Apple Is Warming Up To Social Media: Apple is ...
3801         5  RT @shannonmmiller: Love the @Apple is support...
3802         5  @marcbulandr I could not agree more. Between @...
3803         5  RT @SwiftKey: We're so excited to be named to ...

[1642 rows x 2 columns]


In [ ]:
data['sentiment']=data['sentiment'].map({'5':1,'1':0})
print("Mapped Sentiment values:\n",data['sentiment'].value_counts())

Mapped Sentiment values:
 0    1219
1     423
Name: sentiment, dtype: int64


In [ ]:
data

,sentiment,text
0,0,WTF MY BATTERY WAS 31% ONE SECOND AGO AND NOW ...
1,0,@apple Contact sync between Yosemite and iOS8 ...
2,0,WARNING IF YOU BUY AN IPHONE 5S UNLOCKED FROM ...
3,0,"@Apple, For the love of GAWD, CENTER the '1'on..."
4,0,i get the storage almost full notification lit...
...,...,...
3799,1,Lets go shopping!!! @apple http://t.co/uTFUHuoJIi
3800,1,Apple Is Warming Up To Social Media: Apple is ...
3801,1,RT @shannonmmiller: Love the @Apple is support...
3802,1,@marcbulandr I could not agree more. Between @...


In [ ]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1642 entries, 0 to 3803
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  1642 non-null   int64 
 1   text       1642 non-null   object
dtypes: int64(1), object(1)
memory usage: 38.5+ KB
None


In [ ]:
majority_class=data[data['sentiment']==0]
minority_class=data[data['sentiment']==1]

manjority_class_undersampled=majority_class.sample(len(minority_class))
data=pd.concat([manjority_class_undersampled,minority_class])
print(data['sentiment'].value_counts())

0    423
1    423
Name: sentiment, dtype: int64


In [ ]:
#TEXT Preprocessing

In [ ]:
tokenizer=Tokenizer(num_words=10000,oov_token="<OOV>")
tokenizer.fit_on_texts(data['text'])
sequences=tokenizer.texts_to_sequences(data['text'])

padded_sequences=pad_sequences(sequences,maxlen=50)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(padded_sequences,data['sentiment'],test_size=0.2,random_state=42)

In [ ]:
print("X-train shape:",X_train)
print("X-test shape:",X_test)
print("Y-train shape:",y_train)
print("Y-test shape:",y_test)

X-train shape: [[   0    0    0 ... 1116  276 1117]
 [   0    0    0 ...  452 2770 2771]
 [   0    0    0 ...   55 2624   88]
 ...
 [   0    0    0 ...  205    2  138]
 [   0    0    0 ...    7   66   21]
 [   0    0    0 ... 1424   50 1425]]
X-test shape: [[   0    0    0 ...    3    4 1276]
 [   0    0    0 ...    3    4   79]
 [   0    0    0 ...    4 1935    2]
 ...
 [   0    0    0 ... 3059   92 1144]
 [   0    0    0 ...  691    2  835]
 [   0    0    0 ...    3    4 1924]]
Y-train shape: 3623    1
3591    1
3528    1
3746    1
3497    1
       ..
1069    0
764     0
462     0
3393    1
781     0
Name: sentiment, Length: 676, dtype: int64
Y-test shape: 506     0
486     0
869     0
1130    0
495     0
       ..
3750    1
511     0
3718    1
3736    1
769     0
Name: sentiment, Length: 170, dtype: int64


In [ ]:
model=Sequential([
    Embedding(10000,64,input_length=50),
    LSTM(64,return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(X_train,y_train,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
17/17 [==============================] - 19s 288ms/step - loss: 0.6915 - accuracy: 0.5722 - val_loss: 0.6861 - val_accuracy: 0.6618
Epoch 2/10
17/17 [==============================] - 3s 153ms/step - loss: 0.6531 - accuracy: 0.7593 - val_loss: 0.5961 - val_accuracy: 0.7574
Epoch 3/10
17/17 [==============================] - 3s 151ms/step - loss: 0.4900 - accuracy: 0.8056 - val_loss: 0.5124 - val_accuracy: 0.7721
Epoch 4/10
17/17 [==============================] - 3s 173ms/step - loss: 0.2532 - accuracy: 0.9259 - val_loss: 0.5269 - val_accuracy: 0.7868
Epoch 5/10
17/17 [==============================] - 3s 185ms/step - loss: 0.0974 - accuracy: 0.9741 - val_loss: 0.5569 - val_accuracy: 0.8015
Epoch 6/10
17/17 [==============================] - 3s 173ms/step - loss: 0.0511 - accuracy: 0.9907 - val_loss: 0.6171 - val_accuracy: 0.8088
Epoch 7/10
17/17 [==============================] - 3s 157ms/step - loss: 0.0363 - accuracy: 0.9926 - val_loss: 0.6818 - val_accuracy: 0.7941
Epoch

In [ ]:
y_pred=(model.predict(X_test)>0.5).astype("int32")

6/6 [==============================] - 4s 40ms/step


In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 0.7588235294117647
